In [ ]:
pip install pandas requests selenium webdriver-manager selenium-stealth

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached selenium-4.44.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached selenium_stealth-1.0.6-py3-none-any.whl.metadata (6.4 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.15-py3-none-any.whl.metadata (7.7 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached at

In [ ]:
import os
import time
import random
import pandas as pd
import requests
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By  # 중요! By를 불러옵니다.
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- [1. 설정 및 경로] ---
CLIENT_ID = "xaycKafg5L6rPQFiuw0x"
CLIENT_SECRET = "nYRS3j1EhY"
BASE_DIR = os.getcwd() 
SAVE_FILE = os.path.join(BASE_DIR, "naver_reviews_final_100k.csv") 
LOG_FILE = os.path.join(BASE_DIR, "collected_books_log.txt")

# 수집 설정
TARGET_CATEGORIES = ["소설", "경영", "경제", "자기계발", "인문", "사회", "역사", "과학", "IT", "예술", "여행", "어린이", "만화"]
SAVE_INTERVAL = 60 # 1분마다 파일 물리적 저장

# 시작 시 빈 파일이라도 생성
if not os.path.exists(SAVE_FILE):
    pd.DataFrame(columns=['도서명', '리뷰내용', '카테고리', '수집일자']).to_csv(SAVE_FILE, index=False, encoding='utf-8-sig')

def setup_driver():
    options = Options()
    options.add_argument('--headless=new') # 백그라운드 실행
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2}) # 이미지 미로딩
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

def main():
    driver = setup_driver()
    total_review_count = 0
    last_save_time = time.time()
    pending_reviews = [] 

    # 중복 체크 로그 로드
    collected_links = set()
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r', encoding='utf-8') as f:
            collected_links = set(line.strip() for line in f)

    print(f"🚀 [시작] 10만 건 목표 수집 엔진 가동!")
    print(f"📍 저장 위치: {SAVE_FILE}")

    try:
        for cat in TARGET_CATEGORIES:
            print(f"\n📂 카테고리 [{cat}] 분석 시작...")
            
            for start in range(1, 1001, 100):
                api_url = f"https://openapi.naver.com/v1/search/book.json?query={cat}&display=100&start={start}"
                headers = {"X-Naver-Client-Id": CLIENT_ID, "X-Naver-Client-Secret": CLIENT_SECRET}
                
                try:
                    res = requests.get(api_url, headers=headers).json()
                    items = res.get('items', [])
                    if not items: break
                except: break

                for item in items:
                    link = item['link']
                    title = item['title'].replace('<b>', '').replace('</b>', '')
                    if link in collected_links: continue

                    try:
                        driver.get(link)
                        # 페이지 로딩 대기 (By.CSS_SELECTOR로 수정됨)
                        time.sleep(random.uniform(1.5, 2.5))
                        
                        # 네이버 리뷰 구역 찾기 (클래스명 다중 대응)
                        review_elements = driver.find_elements(By.CSS_SELECTOR, "div[class*='ReviewItem_text'], .comment_text_box, .txt_desc")
                        
                        for el in review_elements:
                            text = el.text.strip()
                            if len(text) > 10:
                                pending_reviews.append({
                                    '도서명': title,
                                    '리뷰내용': text,
                                    '카테고리': cat,
                                    '수집일자': time.strftime('%Y-%m-%d')
                                })
                        
                        # 로그 기록
                        collected_links.add(link)
                        with open(LOG_FILE, 'a', encoding='utf-8') as f:
                            f.write(link + "\n")

                        # --- [1분 주기 실시간 저장] ---
                        if (time.time() - last_save_time) >= SAVE_INTERVAL and pending_reviews:
                            with open(SAVE_FILE, 'a', newline='', encoding='utf-8-sig') as f:
                                writer = csv.DictWriter(f, fieldnames=['도서명', '리뷰내용', '카테고리', '수집일자'])
                                writer.writerows(pending_reviews)
                            
                            total_review_count += len(pending_reviews)
                            print(f" 💾 [자동 저장] 현재까지 총 {total_review_count}건 확보 완료")
                            pending_reviews = []
                            last_save_time = time.time()
                        
                    except Exception as e:
                        print(f" 🔍 {title[:10]}... 분석 중 건너뜀")
                        continue
                    
                    time.sleep(random.uniform(0.4, 0.8))

    except Exception as global_e:
        print(f"❌ 예기치 못한 에러: {global_e}")
    finally:
        # 종료 전 잔여 데이터 저장
        if pending_reviews:
            with open(SAVE_FILE, 'a', newline='', encoding='utf-8-sig') as f:
                writer = csv.DictWriter(f, fieldnames=['도서명', '리뷰내용', '카테고리', '수집일자'])
                writer.writerows(pending_reviews)
            total_review_count += len(pending_reviews)
        
        driver.quit()
        print(f"🏁 수집 종료! 총 {total_review_count}건 저장됨.")

if __name__ == "__main__":
    main()

🚀 [시작] 10만 건 목표 수집 엔진 가동!
📍 저장 위치: c:\Users\이동현\Desktop\웹크롤링\naver_reviews_final_100k.csv

📂 카테고리 [소설] 분석 시작...
 💾 [자동 저장] 현재까지 총 5건 확보 완료
